# 04 · Advanced A2A Handoff + Codex App Server
현장 담당 Agent의 구매요청, 신규 원부자재 생산 투입, Cost Center 변경 handoff를 Orchestration Agent가 받아, LLM이 Team delegation·A2A handoff·SAP BTP Grounding 조회를 선택하는 실습입니다.

**학습 목표:** LLM Orchestration의 delegation·A2A Message·Task·Artifact·middleware와 SAP BTP Grounding 근거를 확인합니다.  
**실행 경계:** 조직 Team과 Codex SQL Generator Agent를 A2A Card로 공개합니다. Team 내부 sub-agent와 다음 Agent 선택은 LLM이 입력과 누적 artifact로부터 생성하며, 이 실습은 ERP·SAP에 실제 쓰기 작업을 하지 않습니다. Codex SQL Generator Agent도 읽기 전용·무승인으로만 실행됩니다.



In [2]:
from pathlib import Path
import sys
from IPython.display import HTML, display

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

from src.agent_to_agent_advanced.ui import ADVANCED_A2A_WORKFLOW_GRAPH_SVG

display(HTML(ADVANCED_A2A_WORKFLOW_GRAPH_SVG))


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  A2A 서버 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">A2A Protocol 1.0 JSON-RPC Task</h2>
<p style="margin: 7px 0;">먼저 별도 터미널에서 <code>python 4_agent_patterns/main.py fnb-a2a-advanced-local</code>을 실행하세요. Advanced 전용 Agent Card와 A2A JSON-RPC Task를 준비합니다. 이 서버는 03 실습의 Agent 서버와 다른 포트를 사용하므로 동시에 실행할 수 있습니다.</p>
  </div>
</div>


In [1]:
import importlib
import src.agent_to_agent_advanced.catalog as a2a_catalog_module
import src.agent_to_agent_advanced.orchestration as a2a_module
import src.agent_to_agent_advanced.ui as a2a_ui_module
a2a_catalog_module = importlib.reload(a2a_catalog_module)
a2a_module = importlib.reload(a2a_module)
a2a_ui_module = importlib.reload(a2a_ui_module)
DEFAULT_REQUEST = a2a_catalog_module.DEFAULT_REQUEST
SCENARIO_REQUESTS = a2a_catalog_module.SCENARIO_REQUESTS
list_agents = a2a_module.list_agents
run_handoff = a2a_module.run_handoff
if 'a2a_ui' in globals():
    a2a_ui.close()
a2a_ui = a2a_ui_module.launch_a2a_ui()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>네 개의 <code>/.well-known/agent-card.json</code> Card에서 조직 경계와 Skill·접속 URL을 확인한다.</li><li>구매팀, 제조기획 Agent, IT팀 Agent가 선택될 때의 TextPart·JSON DataPart와 Task ID를 확인한다.</li><li>제조기획 Agent의 생산기획·OPEX 예산·Cost Center·예산 이관 신청 검토와 IT팀 Agent의 ERP 데이터·변경관리 검토 artifact를 확인한다.</li><li>사람 승인과 외부 시스템 권한 없이는 예산 이관이나 ERP 반영을 수행하지 않는 경계를 확인한다.</li></ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
    <h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">언제 A2A를 사용할까?</h2>
    <p style="margin: 7px 0;">고정된 내부 Agent를 한 팀이 관리한다면 일반 HTTPS delegation만으로 충분합니다. URL, 요청·응답 JSON, 인증, 재시도, 작업 상태를 팀의 API 계약으로 일관되게 관리하면 A2A는 필수가 아닙니다.</p>
    <table style="width:100%; margin:10px 0 4px; border-collapse:collapse; font-size:.92em;"><thead><tr><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">필요</th><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">A2A 기능</th></tr></thead><tbody><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">무슨 Agent인지·무엇을 하는지 발견</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Agent Card</b>의 Skill, 입출력 모드, 접속 URL</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">장기 작업의 상태·실패·취소 관리</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Task</b> ID와 working / completed / failed 등 표준 lifecycle</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">다음 Agent에 구조화 결과 전달</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Message·Artifact·DataPart</b>의 JSON·파일 전달 계약</td></tr><tr><td style="padding:7px;">서로 다른 팀·프레임워크 Agent 연결</td><td style="padding:7px;"><b>JSON-RPC/HTTP</b> 공통 계약으로 내부 구현을 숨긴 상호운용</td></tr></tbody></table>
    <p style="margin: 9px 0 0; color:#4b5f78;"><b>판단:</b> 내부의 고정된 3개 Agent라면 HTTPS handoff가 더 단순할 수 있습니다. 외부·타팀 Agent, 교체 가능한 Agent, 여러 프레임워크 연결이 필요할 때 A2A의 표준화 비용 절감 효과가 커집니다.</p>
  </div>
</div>


In [ ]:
from src.agent_to_agent_advanced.catalog import DEFAULT_REQUEST
from src.agent_to_agent_advanced.orchestration import list_agents, run_handoff


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. A2A Team Card와 내부 sub-agent 보기</h2><p style="margin: 7px 0;">A2A는 조직 경계만 공개합니다. 설비팀 안에는 현장·엔지니어·관리자 Agent, 구매팀 안에는 구매·원가·재고·MDM Agent가 있습니다. 제조기획 Agent는 생산기획·OPEX 예산·Cost Center 관리·예산 이관 신청 Agent를, IT팀 Agent는 ERP 데이터·IT 변경관리 Agent를 조율합니다.</p></div></div>


In [ ]:
for agent in list_agents():
    print(f"## {agent['agent_label']} ({agent['agent_id']})")
    print(f"설명: {agent['description']}")
    print(f"Skill: {agent['skill_id']} | URL: {agent['url']}")
    print(f"내부 sub-agent: {' → '.join(agent['sub_agents'])}")
    print(f"실행 경계: {agent['execution_boundary']}\n")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 운영변경에서 ERP 데이터 반영 준비까지</h2><p style="margin: 7px 0;">UI의 업무 시나리오는 설비 예비품 구매, 신규 원부자재의 생산계획 투입, 생산라인 Cost Center 변경을 제공합니다. LLM은 입력과 누적 artifact를 바탕으로 조달·생산계획·예산·ERP 준비 관점을 선택하며, 실제 ERP API는 호출하지 않습니다.</p></div></div>


In [ ]:
request = DEFAULT_REQUEST  # 필요하면 여기에서 실제 현장 장애 설명으로 바꾸세요.
result = run_handoff(request)
print('[A2A Context]', result['context_id'])
print('[종료 상태]', result['termination'])
print('\n[Team artifact]')
for record in result['artifacts']:
    synthesis = record['artifact']['payload'].get('team_synthesis', {})
    print(f"- {record['producer']}: {synthesis.get('summary', '분석 완료')}")


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #10a37f; background: #edf9f5; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #0b7a5d; letter-spacing: .02em;">🤖 ADVANCED · CODEX APP SERVER</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. A2A artifact를 Codex 분석 Agent에 전달하기</h2><p style="margin: 7px 0;">앞 단계의 A2A Team artifact를 로컬 <code>codex app-server --stdio</code>에 JSON-RPC로 전달합니다. App Server는 <code>initialize → thread/start → turn/start</code> 순서로 연결하고, <code>agentMessage/delta</code>와 <code>turn/completed</code> notification을 스트리밍합니다. 아래 예제는 <code>read-only</code> sandbox로 실행되며 파일을 변경하지 않습니다.</p><p style="margin: 7px 0 0; color:#4b5f78;">사전 조건: <code>codex</code> CLI 설치 및 로그인. 프로토콜 참고: <a href="https://github.com/openai/codex/blob/main/codex-rs/app-server/README.md" target="_blank">OpenAI Codex App Server README</a></p></div></div>


In [ ]:
import json
import shutil
import subprocess
from itertools import count
from queue import Empty, Queue
from threading import Thread
from time import monotonic

class CodexAppServer:
    """Minimal JSON-RPC client for a local `codex app-server --stdio` process."""
    def __init__(self):
        codex = shutil.which('codex')
        if not codex:
            raise RuntimeError('codex CLI를 찾을 수 없습니다. Codex CLI를 설치하고 로그인하세요.')
        self.process = subprocess.Popen(
            [codex, 'app-server', '--stdio'],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, bufsize=1,
        )
        self._messages, self._events, self._request_ids = Queue(), [], count(1)
        Thread(target=self._read_stdout, daemon=True).start()

    def _read_stdout(self):
        for line in self.process.stdout:
            try:
                self._messages.put(json.loads(line))
            except json.JSONDecodeError:
                pass  # stdout에는 JSON-RPC message만 사용합니다.

    def _send(self, message):
        if self.process.poll() is not None:
            raise RuntimeError('Codex App Server가 종료되었습니다.')
        self.process.stdin.write(json.dumps(message) + '\n')
        self.process.stdin.flush()

    def request(self, method, params=None, timeout=30):
        request_id = next(self._request_ids)
        self._send({'id': request_id, 'method': method, 'params': params or {}})
        deadline = monotonic() + timeout
        while True:
            try:
                message = self._messages.get(timeout=max(0.1, deadline - monotonic()))
            except Empty as error:
                raise TimeoutError(f'{method} 응답을 {timeout}초 안에 받지 못했습니다.') from error
            if message.get('id') != request_id:
                self._events.append(message)
                continue
            if 'error' in message:
                raise RuntimeError(f'{method} 실패: {message["error"]}')
            return message['result']

    def initialize(self):
        result = self.request('initialize', {
            'clientInfo': {'name': 'a2a-advanced-notebook', 'version': '1.0'},
            'capabilities': {},
        })
        self._send({'method': 'initialized'})
        return result

    def next_event(self, timeout=120):
        if self._events:
            return self._events.pop(0)
        try:
            return self._messages.get(timeout=timeout)
        except Empty as error:
            raise TimeoutError(f'App Server event를 {timeout}초 안에 받지 못했습니다.') from error

    def close(self):
        if self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self.process.kill()


In [ ]:
if 'codex_server' in globals():
    codex_server.close()

codex_server = CodexAppServer()
server_info = codex_server.initialize()
print(f"Connected: {server_info['userAgent']} ({server_info['platformOs']})")


In [ ]:
# thread/start: 실제 파일 변경을 막기 위해 ephemeral + read-only + never approval을 사용합니다.
thread = codex_server.request('thread/start', {
    'cwd': str(project_root),
    'ephemeral': True,
    'sandbox': 'read-only',
    'approvalPolicy': 'never',
})['thread']

artifacts = result.get('artifacts', []) if 'result' in globals() else []
artifact_summary = '\n'.join(
    f"- {record['producer']}: {record['artifact']['payload'].get('team_synthesis', {}).get('summary', '분석 완료')}"
    for record in artifacts
) or '- A2A handoff 결과를 먼저 실행한 뒤 다시 실행하세요.'
prompt = f"""
다음은 F&B 운영변경 A2A handoff의 Team artifact입니다.
{artifact_summary}

위 결과에서 사람이 승인해야 할 항목, ERP 반영 전에 확인할 항목, 누락된 근거를 한국어로 간결하게 검토하세요.
분석만 수행하고 파일 변경, 명령 실행, 외부 시스템 호출은 하지 마세요.
""".strip()
turn = codex_server.request('turn/start', {
    'threadId': thread['id'],
    'input': [{'type': 'text', 'text': prompt}],
})['turn']

# notification을 소비합니다. delta는 진행 중인 응답, turn/completed는 terminal event입니다.
while True:
    event = codex_server.next_event()
    method, params = event.get('method'), event.get('params', {})
    if params.get('threadId') != thread['id']:
        continue
    if method == 'agentMessage/delta':
        print(params.get('delta', ''), end='', flush=True)
    elif method == 'item/completed' and params.get('item', {}).get('type') == 'agentMessage':
        print(params['item'].get('text', ''))
    elif method == 'turn/completed' and params.get('turn', {}).get('id') == turn['id']:
        print(f"\n[turn completed] {params['turn'].get('status')}")
        break


In [ ]:
# 실습이 끝나면 로컬 App Server 프로세스를 종료합니다.
codex_server.close()


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #10a37f; background: #edf9f5; border-radius: 6px; color: #202124;"><div style="font-size: .82em; font-weight: 700; color: #0b7a5d; letter-spacing: .02em;">🤖 ADVANCED · CODEX APP SERVER AGENT</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Codex SQL Generator Agent 등록하기</h2><p style="margin: 7px 0;">Codex SQL Generator Agent는 <code>http://127.0.0.1:9223</code>의 Agent Card와 A2A Task를 제공합니다. 제공된 SQL dialect·스키마·A2A artifact만 근거로 단일 읽기 전용 <code>SELECT</code> 또는 <code>WITH … SELECT</code> 초안을 생성하며 SQL을 실행하지 않습니다.</p><p style="margin: 7px 0 0; color:#4b5f78;">App Server는 task마다 <code>ephemeral</code>, <code>read-only</code>, <code>approvalPolicy: never</code>로 실행됩니다. INSERT·UPDATE·DELETE·DDL, 파일 변경·명령 실행·외부 시스템 호출은 Agent system prompt와 sandbox에서 금지됩니다. 사전 조건: <code>codex</code> CLI 설치 및 로그인.</p></div></div>


In [ ]:
from src.agent_to_agent_advanced.codex_app_server import CODEX_SQL_GENERATOR_AGENT_ID

codex_agent = next(agent for agent in list_agents() if agent['agent_id'] == CODEX_SQL_GENERATOR_AGENT_ID)
print(f"등록됨: {codex_agent['agent_label']} | {codex_agent['url']}")
if 'result' not in globals():
    print('먼저 2번 A2A handoff를 실행하세요.')
else:
    reviews = [record for record in result['artifacts'] if record['producer'] == CODEX_SQL_GENERATOR_AGENT_ID]
    if not reviews:
        print('이번 요청에서는 LLM이 Codex SQL Generator Agent가 필요하지 않다고 판단했습니다.')
    else:
        generated = reviews[-1]['artifact']['payload']['sql_generation']
        print('\n[Codex SQL Generator Agent]')
        print(generated['sql'] or '생성 불가: ' + ', '.join(generated['missing_inputs']))
        print('안전 경계:', generated['safety_notes'])
